# Summary_Day10_offline.ipynb  
## 다중 분류 2 · 인터넷 불가 버전 · sklearn digits 대체 실습

이 파일은 **인터넷이 안 되는 환경**을 기준으로 만든 10강 실습 노트다.

강의 기준 데이터는 MNIST다.  
하지만 MNIST는 보통 `torchvision.datasets.MNIST(download=True)`로 내려받아야 하므로 인터넷이 필요하다.  
인터넷이 안 되는 환경에서는 scikit-learn 내장 데이터인 `load_digits()`를 사용해 같은 흐름을 연습한다.

```text
MNIST: 28 × 28 이미지 → 784 feature
sklearn digits: 8 × 8 이미지 → 64 feature
```

크기는 다르지만 학습 흐름은 같다.

```text
이미지 픽셀
→ Flatten
→ TensorDataset
→ DataLoader
→ MLP
→ CrossEntropyLoss
→ torch.max로 예측
→ classification_report / confusion_matrix
```

> 필기 포인트:  
> 인터넷이 없을 때 중요한 것은 “MNIST와 완전히 같은 데이터”가 아니라, 강의에서 배운 이미지 분류 구조를 끊기지 않고 연습하는 것이다.

## 1. 인터넷 불가 버전의 전체 목적

이 노트북의 목적은 다음이다.

1. 인터넷 없이 내장 digits 데이터를 불러온다.
2. 8×8 이미지를 64개 feature로 펼친다.
3. `TensorDataset`과 `DataLoader`로 mini-batch를 만든다.
4. MLP로 숫자 0~9를 다중 분류한다.
5. `CrossEntropyLoss`와 `torch.max(outputs, 1)[1]` 패턴을 복습한다.
6. macro/micro/weighted 평균 지표를 확인한다.
7. 클래스 불균형 상황에서 class weight와 sampler 개념을 정리한다.

## 2. 라이브러리 준비

### 함수/모듈 사용법

```python
from sklearn.datasets import load_digits
```

- 인터넷 없이 사용할 수 있는 손글씨 숫자 데이터다.
- 이미지 크기는 8×8이다.
- class는 0~9까지 10개다.

```python
TensorDataset(X, y)
DataLoader(dataset, batch_size=...)
```

- 이미 준비된 Tensor를 dataset처럼 묶고, mini-batch로 꺼낼 수 있게 만든다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

from sklearn.datasets import load_digits, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    roc_auc_score,
)

%matplotlib inline

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("PyTorch:", torch.__version__)

## 3. digits 데이터 불러오기

### 함수 사용법

```python
digits = load_digits()
```

- `digits.images`: `[N, 8, 8]` 이미지 배열이다.
- `digits.data`: `[N, 64]`로 이미 펼쳐진 feature 배열이다.
- `digits.target`: 0~9 정수 label이다.

In [ ]:
digits = load_digits()

X_images = digits.images
X_flat = digits.data
y = digits.target

print("X_images shape:", X_images.shape)
print("X_flat shape:", X_flat.shape)
print("y shape:", y.shape)
print("class 개수:", len(np.unique(y)))
print("class 분포:", np.bincount(y))

출력 해석:

```text
X_images = [1797, 8, 8]
X_flat = [1797, 64]
```

- 1797개 이미지가 있다.
- 한 이미지는 8×8이다.
- Flatten하면 64개 feature가 된다.

## 4. 이미지 확인하기

8×8 이미지를 직접 시각화한다.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(8, 4))

for ax, img, label in zip(axes.ravel(), X_images[:10], y[:10]):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"label={label}")
    ax.axis("off")

plt.tight_layout()
plt.show()

그래프 해석:

- MNIST보다 해상도는 낮지만 손글씨 숫자 이미지다.
- 컴퓨터는 이 이미지를 8×8 숫자 격자로 본다.
- MLP에 넣을 때는 64개 숫자로 펼쳐 사용한다.

## 5. 픽셀 스케일 조정

digits 픽셀값은 0~16 범위다.  
간단히 16으로 나누어 0~1 범위로 만든다.

```python
X = X_flat / 16.0
```

> MNIST에서 `ToTensor()`가 0~255를 0~1로 바꾸는 것과 비슷한 역할이다.

In [ ]:
X = X_flat / 16.0

print("X min:", X.min())
print("X max:", X.max())
print("X shape:", X.shape)

## 6. Train / Test 분할

### 함수 사용법

```python
train_test_split(X, y, test_size=0.2, stratify=y)
```

- `test_size=0.2`: 전체의 20%를 테스트로 둔다.
- `stratify=y`: class 비율을 유지한다.
- 다중 분류에서 class 분포를 유지하는 것이 중요하다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("train class 분포:", np.bincount(y_train))
print("test class 분포:", np.bincount(y_test))

## 7. Tensor 변환

다중 분류에서는 label dtype이 중요하다.

```python
X_tensor = torch.FloatTensor(X)
y_tensor = torch.LongTensor(y)
```

- 입력 feature는 float다.
- 정답 label은 long이다.
- `CrossEntropyLoss`는 label을 class index로 받기 때문이다.

In [ ]:
X_train_t = torch.FloatTensor(X_train)
X_test_t = torch.FloatTensor(X_test)

y_train_t = torch.LongTensor(y_train)
y_test_t = torch.LongTensor(y_test)

print("X_train_t:", X_train_t.shape, X_train_t.dtype)
print("y_train_t:", y_train_t.shape, y_train_t.dtype)

## 8. TensorDataset과 DataLoader 만들기

인터넷 없이 직접 배열을 갖고 있을 때는 `TensorDataset`을 사용하면 된다.

### 함수 사용법

```python
TensorDataset(X_train_t, y_train_t)
DataLoader(dataset, batch_size=64, shuffle=True)
```

- `TensorDataset`: 입력과 label을 한 쌍으로 묶는다.
- `DataLoader`: batch 단위로 꺼낸다.

In [ ]:
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("train batch 개수:", len(train_loader))
print("test batch 개수:", len(test_loader))

## 9. Batch shape 확인

mini-batch 하나를 꺼내 shape을 확인한다.

In [ ]:
batch_X, batch_y = next(iter(train_loader))

print("batch_X shape:", batch_X.shape)
print("batch_y shape:", batch_y.shape)
print("첫 번째 label:", batch_y[0].item())

출력 해석:

```text
batch_X shape = [64, 64]
```

- 첫 번째 64는 batch size다.
- 두 번째 64는 8×8 이미지를 펼친 feature 개수다.

## 10. 4차원 이미지 Tensor로도 바꿔보기

CNN에서는 Flatten하지 않고 이미지 공간 구조를 유지한다.

digits 이미지는 CNN 기준으로 다음 shape이 된다.

```text
[N, 1, 8, 8]
```

### 함수 사용법

```python
torch.FloatTensor(X_images).unsqueeze(1)
```

- `unsqueeze(1)`은 channel 차원을 추가한다.

In [ ]:
X_images_scaled = X_images / 16.0
X_images_4d = torch.FloatTensor(X_images_scaled).unsqueeze(1)

print("CNN용 4D shape:", X_images_4d.shape)

> 지금 실습은 MLP라서 `[N, 64]`를 사용한다.  
> 다음 CNN 강의에서는 `[N, 1, 8, 8]` 같은 4차원 입력이 중요해진다.

## 11. Offline MLP 모델 정의

8×8 이미지를 펼친 64개 feature를 입력으로 사용한다.

구조는 다음이다.

```text
Linear(64 → 64)
→ ReLU
→ Linear(64 → 10)
```

In [ ]:
class OfflineDigitsMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.network(x)

model = OfflineDigitsMLP().to(device)

print(model)

### 함수 사용법 정리

```python
nn.Linear(64, 64)
```

- 64개 입력 feature를 64개 은닉 표현으로 바꾼다.

```python
nn.Linear(64, 10)
```

- 숫자 0~9 class 점수 10개를 출력한다.

마지막에는 Softmax를 붙이지 않는다.  
`CrossEntropyLoss`가 내부에서 처리한다.

## 12. 손실함수와 Optimizer

다중 분류이므로 `CrossEntropyLoss`를 사용한다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("criterion:", criterion)
print("optimizer:", optimizer.__class__.__name__)

## 13. 학습/평가 함수 만들기

온라인 버전과 같은 패턴이다.

```text
train: model.train() + backward + step
eval: model.eval() + no_grad
```

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pred = torch.max(outputs, 1)[1]
        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            pred = torch.max(outputs, 1)[1]

            correct += (pred == y_batch).sum().item()
            total += y_batch.size(0)

            all_labels.extend(y_batch.cpu().numpy())
            all_preds.extend(pred.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return (
        total_loss / len(loader),
        correct / total,
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs)
    )

## 14. 모델 학습 실행

인터넷 없이도 바로 실행되는 학습이다.

In [ ]:
num_epochs = 20

history = {
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": []
}

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, _, _, _ = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    if (epoch + 1) % 5 == 0:
        print(
            f"epoch {epoch + 1} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"test_loss={test_loss:.4f} | test_acc={test_acc:.4f}"
        )

## 15. 학습 곡선 확인

Loss와 Accuracy를 확인한다.

In [ ]:
plt.plot(history["train_loss"], label="train loss")
plt.plot(history["test_loss"], label="test loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Offline Digits Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(history["train_acc"], label="train acc")
plt.plot(history["test_acc"], label="test acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Offline Digits Accuracy")
plt.legend()
plt.show()

그래프 해석:

- train/test loss가 함께 내려가면 학습이 잘 진행되는 것이다.
- accuracy가 올라가면 숫자 class를 더 많이 맞히는 것이다.
- train만 좋아지고 test가 나빠지면 과적합을 의심한다.

## 16. Classification Report와 평균 방식

강의 후반부에서는 precision, recall, f1-score와 macro/micro/weighted 평균이 나온다.

### 평균 방식 정리

| 평균 | 의미 |
|---|---|
| micro | 전체 샘플 기준으로 합산해서 계산한다 |
| macro | class별 점수를 단순 평균낸다 |
| weighted | class별 sample 수를 반영해 평균낸다 |

불균형 데이터에서는 macro와 weighted를 같이 보는 것이 중요하다.

In [ ]:
test_loss, test_acc, y_true, y_pred, y_prob = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print("test accuracy:", test_acc)

print(classification_report(
    y_true,
    y_pred,
    target_names=[f"digit {i}" for i in range(10)],
    zero_division=0
))

print("macro precision:", precision_score(y_true, y_pred, average="macro", zero_division=0))
print("macro recall:", recall_score(y_true, y_pred, average="macro", zero_division=0))
print("macro f1:", f1_score(y_true, y_pred, average="macro", zero_division=0))
print("weighted f1:", f1_score(y_true, y_pred, average="weighted", zero_division=0))

## 17. Confusion Matrix 확인

다중 분류에서는 어떤 숫자끼리 헷갈리는지 확인하는 것이 중요하다.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.imshow(cm)
plt.title("Offline Digits Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for (i, j), value in np.ndenumerate(cm):
    if value > 0:
        plt.text(j, i, str(value), ha="center", va="center", fontsize=8)

plt.colorbar()
plt.show()

그래프 해석:

- 대각선은 맞힌 개수다.
- 대각선 밖의 값은 헷갈린 숫자다.
- 예를 들어 8과 9가 헷갈리면 해당 칸에 값이 생긴다.

## 18. 다중 분류 ROC-AUC 개념

다중 분류에서 ROC-AUC를 계산하려면 각 class를 “이 class냐 아니냐” 문제로 바꿔야 한다.

이를 one-vs-rest 방식이라고 볼 수 있다.

### 함수 사용법

```python
label_binarize(y, classes=range(10))
```

- 정수 label을 one-hot 형태로 바꾼다.
- ROC curve를 class별로 계산할 때 필요하다.

In [ ]:
classes = np.arange(10)

y_true_bin = label_binarize(y_true, classes=classes)

fpr = {}
tpr = {}
roc_auc = {}

for i in classes:
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

micro_auc = roc_auc_score(y_true_bin, y_prob, average="micro", multi_class="ovr")
macro_auc = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr")

print("micro AUC:", micro_auc)
print("macro AUC:", macro_auc)
print("class 0 AUC:", roc_auc[0])

In [ ]:
for i in range(3):
    plt.plot(fpr[i], tpr[i], label=f"class {i} AUC={roc_auc[i]:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multi-class ROC Curve Example")
plt.legend()
plt.show()

그래프 해석:

- 다중 분류 ROC는 class별로 따로 그릴 수 있다.
- micro 평균은 전체 샘플 단위로 합산하는 느낌이다.
- macro 평균은 class별 성능을 단순 평균내는 느낌이다.

## 19. 클래스 불균형 데이터 만들기

강의 후반부에서는 클래스 불균형도 다룬다.

예를 들어 어떤 class가 대부분이고, 어떤 class는 매우 적은 상황이다.

```text
정상 990개, 이상 10개
```

이 경우 Accuracy만 보면 위험하다.

In [ ]:
X_imb, y_imb = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_classes=4,
    n_clusters_per_class=1,
    weights=[0.70, 0.15, 0.10, 0.05],
    random_state=42
)

print("class 분포:", np.bincount(y_imb))

## 20. Class Weight 계산하기

소수 class에 더 큰 손실 가중치를 주기 위해 class weight를 계산한다.

강의에서 나온 감각은 다음이다.

```text
적은 class일수록 weight를 크게 준다
많은 class일수록 weight를 작게 준다
```

대표 공식은 다음이다.

```text
weight_i = 전체 샘플 수 / (class 개수 × 해당 class 샘플 수)
```

In [ ]:
class_counts = np.bincount(y_imb)
n_samples = len(y_imb)
n_classes = len(class_counts)

class_weights = n_samples / (n_classes * class_counts)

print("class_counts:", class_counts)
print("class_weights:", class_weights)

class_weights_t = torch.FloatTensor(class_weights).to(device)

### 함수 사용법: `CrossEntropyLoss(weight=...)`

```python
nn.CrossEntropyLoss(weight=class_weights)
```

- class별 손실에 가중치를 준다.
- 소수 class를 틀렸을 때 더 큰 penalty를 부여한다.

In [ ]:
criterion_weighted = nn.CrossEntropyLoss(weight=class_weights_t)

print(criterion_weighted)

## 21. WeightedRandomSampler 개념

class weight는 loss에 가중치를 주는 방식이다.

`WeightedRandomSampler`는 데이터를 뽑는 단계에서 소수 class가 더 자주 나오게 하는 방식이다.

### 함수 사용법

```python
WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
```

- `weights`: 각 샘플이 뽑힐 가중치다.
- `replacement=True`: 복원 추출, 즉 중복해서 뽑을 수 있게 한다.
- 소수 class를 oversampling하는 효과가 있다.

In [ ]:
sample_weights = class_weights[y_imb]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

print("sample_weights shape:", sample_weights.shape)
print("앞 10개 sample weight:", sample_weights[:10])
print("sampler 준비 완료")

> 필기 포인트:  
> `replacement=True`는 같은 데이터를 여러 번 뽑을 수 있다는 뜻이다.  
> 소수 class를 다수 class에 맞추기 위해 중복 추출하는 oversampling 구조다.

## 22. 핵심 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 사용법 |
|---|---|---|
| `load_digits` | 인터넷 없는 손글씨 숫자 데이터 | `load_digits()` |
| `X_images` | 이미지 원본 배열 | `[N, 8, 8]` |
| `X_flat` | 펼쳐진 feature 배열 | `[N, 64]` |
| `TensorDataset` | Tensor를 dataset으로 묶음 | `TensorDataset(X, y)` |
| `DataLoader` | mini-batch 생성 | `DataLoader(dataset, batch_size=...)` |
| `batch_size` | 한 번에 학습할 데이터 수 | 64 등 |
| `CrossEntropyLoss` | 다중 분류 손실 | logits + long label |
| `torch.max(outputs, 1)[1]` | 예측 class | indices 사용 |
| `classification_report` | class별 성능 보고서 | precision, recall, f1 |
| `macro` | class별 단순 평균 | class 균형 관점 |
| `micro` | 전체 샘플 기준 합산 | 전체 샘플 관점 |
| `weighted` | class 개수 반영 평균 | sample 수 반영 |
| `label_binarize` | label one-hot 변환 | ROC-AUC 계산용 |
| `class_weight` | class별 손실 가중치 | 적은 class에 큰 값 |
| `WeightedRandomSampler` | 가중치 기반 샘플링 | oversampling 효과 |
| `replacement` | 복원 추출 여부 | 중복 추출 허용 |

## 23. 시험용 요약

```text
인터넷 불가 버전 = sklearn digits로 MNIST 구조를 대체해서 연습한다
```

꼭 기억할 것:

- MNIST는 28×28이라 784 feature다.
- sklearn digits는 8×8이라 64 feature다.
- 크기는 다르지만 이미지 분류 학습 흐름은 같다.
- 이미지는 픽셀 숫자 격자다.
- MLP는 이미지를 펼친 벡터를 입력으로 받는다.
- `TensorDataset`은 Tensor를 dataset처럼 묶는다.
- `DataLoader`는 mini-batch를 만든다.
- 다중 분류 label은 `torch.long` 타입이어야 한다.
- `CrossEntropyLoss`에는 Softmax를 먼저 붙이지 않는다.
- 예측 class는 `torch.max(outputs, 1)[1]`로 구한다.
- 평가할 때는 `model.eval()`과 `torch.no_grad()`를 사용한다.
- `classification_report`는 precision, recall, f1-score를 보여준다.
- macro 평균은 class별 단순 평균이다.
- micro 평균은 전체 샘플 기준으로 합산한다.
- weighted 평균은 class별 sample 수를 반영한다.
- 다중 분류 ROC-AUC는 label을 one-hot으로 바꿔 계산한다.
- 클래스 불균형에서는 accuracy만 보면 위험하다.
- class weight는 소수 class 손실을 크게 만든다.
- WeightedRandomSampler는 소수 class를 더 자주 뽑게 만든다.
- `replacement=True`는 복원 추출, 즉 중복 추출 허용이다.